In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
DATA='../data'; DP=DATA+'/preprocessed'
labels = np.load(f'{DP}/labels.npy'); N=len(labels)
cvsplit = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# ---- out-of-fold predictions for each method ----
def oof_lr(X):
    pred=np.zeros(N,int); prob=np.zeros(N)
    for tr,te in cvsplit.split(X, labels):
        lr=LogisticRegression(max_iter=2000).fit(X[tr], labels[tr])
        pred[te]=lr.predict(X[te]); prob[te]=lr.predict_proba(X[te])[:,1]
    return pred, prob

methods={}
# baselines (recomputed OOF; fast)
p,pr=oof_lr(np.load(f'{DATA}/baseline_features.npy'));        methods['Handcrafted + LR']=(p,pr)
p,pr=oof_lr(np.load(f'{DATA}/brainomni_features_tiny.npy'));  methods['Frozen tiny + LR']=(p,pr)
p,pr=oof_lr(np.load(f'{DATA}/brainomni_features_base.npy'));  methods['Frozen base + LR']=(p,pr)
# fine-tune OOF from the saved runs (last-1 block)
for tag,fn in [('tiny','finetune_shared.pt'),('base','finetune_shared_base.pt')]:
    if os.path.exists(f'{DP}/{fn}'):
        allcv=torch.load(f'{DP}/{fn}', map_location='cpu', weights_only=False)['cv']
        for nf,lbl in [(10,'last1'),(9,'last2'),(8,'last3')]:
            cv=allcv[nf]
            methods[f'FT {tag} NN {lbl}']=(cv['nn_pred'], cv['nn_prob'])
            methods[f'FT {tag} LR {lbl}']=(cv['lr_pred'], cv['lr_prob'])
print('methods:', list(methods.keys()))

In [ ]:
# ---- bootstrap: 95% CI on accuracy and AUC ----
def boot(y, a, metric, n_boot=2000, seed=0):
    rng=np.random.default_rng(seed); idx=np.arange(N); v=[]
    for _ in range(n_boot):
        b=rng.choice(idx, size=N, replace=True)
        try: v.append(metric(labels[b], a[b]))
        except ValueError: pass   # skip resample with one class only
    return np.mean(v), np.percentile(v,[2.5,97.5])

print(f'{"method":28s}  {"accuracy (95% CI)":24s}  {"AUC (95% CI)"}')
for name,(pred,prob) in methods.items():
    am,aci=boot(labels, pred, accuracy_score)
    um,uci=boot(labels, prob, roc_auc_score)
    print(f'{name:28s}  {am:.3f} [{aci[0]:.3f}, {aci[1]:.3f}]   {um:.3f} [{uci[0]:.3f}, {uci[1]:.3f}]')

method                        accuracy (95% CI)         AUC (95% CI)
Handcrafted + LR              0.872 [0.811, 0.926]   0.924 [0.873, 0.966]
Frozen tiny + LR              0.831 [0.770, 0.892]   0.877 [0.815, 0.933]
Frozen base + LR              0.831 [0.770, 0.892]   0.877 [0.815, 0.933]
Fine-tune tiny + NN (last1)   0.776 [0.709, 0.845]   0.872 [0.810, 0.928]
Fine-tune tiny + LR (last1)   0.790 [0.723, 0.851]   0.873 [0.810, 0.930]


In [ ]:
# ---- paired difference test: is Handcrafted significantly better? ----
def boot_diff(predA, predB, n_boot=2000, seed=0):
    rng=np.random.default_rng(seed); idx=np.arange(N); d=[]
    for _ in range(n_boot):
        b=rng.choice(idx, size=N, replace=True)
        d.append(accuracy_score(labels[b],predA[b]) - accuracy_score(labels[b],predB[b]))
    return np.mean(d), np.percentile(d,[2.5,97.5])

hand=methods['Handcrafted + LR'][0]
print('Handcrafted + LR  vs  each BrainOmni method  (accuracy difference, 95% CI):')
for name,(pred,_) in methods.items():
    if name=='Handcrafted + LR': continue
    m,ci=boot_diff(hand, pred)
    sig='SIGNIFICANT (CI excludes 0)' if (ci[0]>0 or ci[1]<0) else 'not significant (CI includes 0)'
    print(f'  vs {name:28s}: +{m:.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}]  -> {sig}')